In [1]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp0iu2w7gu".


In [4]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>

// 1. THE KERNEL (Runs on GPU)
__global__ void vectorAdd(float* A, float* B, float* C, int n) {
    // Calculate global thread ID
    int i = blockDim.x * blockIdx.x + threadIdx.x;

    // Boundary Check: Ensure we don't access memory outside the array
    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int n = 10000; // Total elements
    size_t size = n * sizeof(float);

    // 2. HOST MEMORY ALLOCATION (CPU)
    float *h_A = (float*)malloc(size);
    float *h_B = (float*)malloc(size);
    float *h_C = (float*)malloc(size);

    // Initialize input data
    for(int i=0; i<n; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    // 3. DEVICE MEMORY ALLOCATION (GPU)
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, size);
    cudaMalloc(&d_B, size);
    cudaMalloc(&d_C, size);

    // 4. HOST TO DEVICE COPY
    // We move the data from CPU RAM to GPU VRAM
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // 5. KERNEL LAUNCH
    // Threads per block (usually 256 or 512)
    int threadsPerBlock = 256;
    // Calculate enough blocks to cover all 'n' elements
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, n);

    // 6. DEVICE TO HOST COPY
    // Bring the result back to CPU to print/verify
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Verify first and last elements
    std::cout << "Verification: C[0] = " << h_C[0] << ", C[n-1] = " << h_C[n-1] << std::endl;

    // 7. CLEANUP
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);

    return 0;
}

Verification: C[0] = 3, C[n-1] = 3

